## Import necessary libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import sys

from sklearn.metrics import r2_score
from model_metrics import (
    summarize_model_performance,
    show_confusion_matrix,
)
from model_tuner import loadObjects
from eda_toolkit import ensure_directory

sys.path.append("../")

from core.functions import (
    plot_cumulative_fatalities_captured_journal,
    load_model_from_mlflow
)

import model_metrics

print(f"Model Metrics version: {model_metrics.__version__}")

## Set Up and Ensure Directories

In [ ]:
base_path = os.path.join(os.pardir)
mlflow_model_path = (
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/306762030449779565"
)
# create image paths
image_path_png = os.path.join(base_path, "images", "png_images")
image_path_svg = os.path.join(base_path, "images", "svg_images")
data_path = os.path.join(base_path, "data")

# Use the function to ensure'data' directory exists
ensure_directory(image_path_png)
ensure_directory(image_path_svg)
ensure_directory(data_path)

## Read in the model objects

In [ ]:
# flavor (run name in MLflow) -> algo prefix used in the artifact folder
FLAVORS = {
    "lr_orig_training":              "lr",
    "lasso_orig_rfe_training":       "lasso",
    "ridge_orig_rfe_training":       "ridge",
    "elastic_net_orig_rfe_training": "elastic_net",
    "xgb_orig_training":             "xgb",
    "cat_orig_training":             "cat",
}

# ----------------------------------------------------------------------
# load all models
# ----------------------------------------------------------------------
models = {
    flavor: load_model_from_mlflow(flavor, algo)
    for flavor, algo in FLAVORS.items()
}

# keep the original short names working for downstream code
model_linear_reg      = models["lr_orig_training"]
model_lasso_rfe       = models["lasso_orig_rfe_training"]
model_ridge_rfe       = models["ridge_orig_rfe_training"]
model_elastic_net_rfe = models["elastic_net_orig_rfe_training"]
model_xgb             = models["xgb_orig_training"]
model_cat             = models["cat_orig_training"]

## Read in partioning data

In [ ]:
X_train = pd.read_parquet("../data/processed/X_train.parquet")
y_train = pd.read_parquet("../data/processed/y_train_log_fatalities.parquet")
X_valid = pd.read_parquet("../data/processed/X_valid.parquet")
y_valid = pd.read_parquet("../data/processed/y_valid_log_fatalities.parquet")
X_test = pd.read_parquet("../data/processed/X_test.parquet")
y_test = pd.read_parquet("../data/processed/y_test_log_fatalities.parquet")

## Read in original data for reference

In [ ]:
df = pd.read_csv("../data/raw/ACLED Data_2026-01-02.csv")

In [ ]:
with open(
    "../data/raw/ACLED Data_2026-01-02.csv", encoding="utf-8", errors="replace",
) as f:
    lines = f.readlines()

print(repr(lines[199]))  # line 201 (0-indexed, plus header = 199)
print(f"\nField count: {lines[199].count(',')}")
print(f"\nHeader count: {lines[0].count(',')}")

In [ ]:
y_pred = model_xgb.predict(X_test)

## Summarize model performance

In [ ]:
categorical_cols = [
    "admin1",
    "sub_event_type",
    "interaction",
    "source_scale",
    "geo_precision",
    "time_precision",
]

In [ ]:
print("Training Set Results")
X_train_raw = X_train
for col in categorical_cols:
    combined_cats = pd.Categorical(
        pd.concat([X_train[col], X_train[col], X_train[col]])
    ).categories
    X_train[col] = pd.Categorical(X_train[col], categories=combined_cats)

train_preds = {
    "Linear Regressor": model_linear_reg.predict(X_train_raw),
    "Lasso RFE": model_lasso_rfe.predict(X_train_raw),
    "XGBoost Regressor": model_xgb.predict(X_train),
    "CatBoost Regressor": model_cat.predict(X_train),
}

for name, pred in train_preds.items():
    print(f"{name} R²: {r2_score(y_train, pred):.4f}")

print()

print("Validation Set Results")
X_valid_raw = X_valid
for col in categorical_cols:
    combined_cats = pd.Categorical(
        pd.concat([X_train[col], X_valid[col], X_valid[col]])
    ).categories
    X_valid[col] = pd.Categorical(X_valid[col], categories=combined_cats)

valid_preds = {
    "Linear Regressor": model_linear_reg.predict(X_valid_raw),
    "Lasso RFE": model_lasso_rfe.predict(X_valid_raw),
    "XGBoost Regressor": model_xgb.predict(X_valid),
    "CatBoost Regressor": model_cat.predict(X_valid),
}

for name, pred in valid_preds.items():
    print(f"{name} R²: {r2_score(y_valid, pred):.4f}")

print()

print("Test Set Results")
X_test_raw = X_test
for col in categorical_cols:
    combined_cats = pd.Categorical(
        pd.concat([X_train[col], X_valid[col], X_test[col]])
    ).categories
    X_test[col] = pd.Categorical(X_test[col], categories=combined_cats)

test_preds = {
    "Linear Regressor": model_linear_reg.predict(X_test_raw),
    "Lasso RFE": model_lasso_rfe.predict(X_test_raw),
    "XGBoost Regressor": model_xgb.predict(X_test),
    "CatBoost Regressor": model_cat.predict(X_test),
}

for name, pred in test_preds.items():
    print(f"{name} R²: {r2_score(y_test, pred):.4f}")

In [ ]:
y_test["fatalities"] = np.expm1(y_test["log_fatalities"])

## Plot Cumulative Fatalities Captured

In [ ]:
plot_cumulative_fatalities_captured_journal(
    y_test["log_fatalities"],
    model_xgb.predict(X_test),
    title="",
    dpi=100,
    model_name="XGBoost",
    save_path="../images/svg_images/cumulative_fatalities_captured.svg",
)

In [ ]:
diff = y_test["fatalities"].values - np.expm1(y_test["log_fatalities"].values)
print(f"Max difference: {np.max(np.abs(diff))}")
print(f"Sum from raw fatalities: {y_test['fatalities'].sum()}")
print(f"Sum from expm1(log_fatalities): {np.expm1(y_test['log_fatalities']).sum()}")

In [ ]:
fig = plot_cumulative_fatalities_captured_journal(
    y_test["log_fatalities"],
    model_xgb.predict(X_test),
    title="",
    dpi=100,
    model_name="XGBoost",
    save_path="../images/svg_images/cumulative_fatalities_captured.svg",
)
fig

## Comparative Analysis: Resource Deployment Efficiency

In [ ]:
# Get test predictions from the updated model
y_test_pred = model_xgb.predict(X_test)

# Sort by predicted severity (descending)
sort_idx = np.argsort(y_test_pred)[::-1]

# Use the ORIGINAL fatalities column (no back-transform needed)
y_test_actual = y_test["fatalities"].values[sort_idx]

# Cumulative fatalities captured
cumulative = np.cumsum(y_test_actual)
total_fatalities = cumulative[-1]
n_events = len(y_test_actual)

# Key percentile thresholds
for pct in [0.01, 0.10, 0.20, 0.50]:
    idx = int(np.ceil(pct * n_events)) - 1
    captured = cumulative[idx] / total_fatalities * 100
    print(
        f"Top {pct*100:.0f}% of events ({idx+1} events): "
        f"{captured:.1f}% of fatalities captured"
    )

print()

# Resource deployment comparison at 20%
top_20_idx = int(np.ceil(0.20 * n_events))
casualties_reached = cumulative[top_20_idx - 1]
casualties_missed = total_fatalities - casualties_reached
eff_model = casualties_reached / top_20_idx
eff_random = total_fatalities / n_events

print(f"Total events: {n_events}")
print(f"Total fatalities: {total_fatalities:.0f}")
print()
print("Baseline (Random Allocation):")
print(f"  Events covered: {top_20_idx} (20%)")
print(f"  Fatalities reached: {total_fatalities * 0.20:.0f} (20%)")
print(f"  Fatalities missed: {total_fatalities * 0.80:.0f} (80%)")
print(f"  Efficiency ratio: {eff_random:.2f} fatalities/event")
print()
print("Model-Guided Allocation:")
print(f"  Events covered: {top_20_idx} (20%)")
print(
    f"  Fatalities reached: {casualties_reached:.0f} ({casualties_reached/total_fatalities*100:.0f}%)"
)
print(
    f"  Fatalities missed: {casualties_missed:.0f} ({casualties_missed/total_fatalities*100:.0f}%)"
)
print(f"  Efficiency ratio: {eff_model:.2f} fatalities/event")
print()
print(f"Improvement over random: {eff_model/eff_random:.1f}x")

In [ ]:
len(model_cat.get_feature_names())

## Plot Actual vs. Predicted Fatalities 

In [ ]:
from core.functions import plot_actual_vs_predicted

plot_actual_vs_predicted(
    y_test["log_fatalities"],
    model_xgb.predict(X_test_raw),
    title="Actual vs Predicted Fatalities",
    save_path="../images/svg_images/actual_vs_predicted",
    save_format="svg",
    show_log_metrics=True,
)

## Summarize Model Performance

In [ ]:
regression_metrics = summarize_model_performance(
    model=[model_linear_reg, model_lasso_rfe, model_xgb, model_cat],
    model_title=[
        "Linear Regressor",
        "Lasso RFE",
        "Elastic Net RFE",
        "XGBoost Regressor",
        "CatBoost Regressor",
    ],
    X=X_valid_raw,
    y=y_valid,
    y_pred=[
        model_linear_reg.predict(X_valid_raw),
        model_lasso_rfe.predict(X_valid_raw),
        model_xgb.predict(X_valid),
        model_cat.predict(X_valid),
    ],
    model_type="regression",
    return_df=True,
    decimal_places=3,
    include_adjusted_r2=True,
)
print("Validation Set Performance Metrics:")
regression_metrics

In [ ]:
X_test.shape

In [ ]:
from model_tuner import report_model_metrics

report_model_metrics(model_xgb, X_test, y_test["log_fatalities"])

In [ ]:
regression_metrics = summarize_model_performance(
    model=[
        model_linear_reg,
        model_lasso_rfe,
        model_elastic_net_rfe,
        model_xgb,
        model_cat,
    ],
    model_title=[
        "Linear Regressor",
        "Lasso RFE",
        "Elastic Net RFE",
        "XGBoost Regressor",
        "CatBoost Regressor",
    ],
    X=X_test_raw,
    y=y_test["log_fatalities"],
    y_pred=[
        model_linear_reg.predict(X_test_raw),
        model_lasso_rfe.predict(X_test_raw),
        model_xgb.predict(X_test),
        model_cat.predict(X_test),
    ],
    model_type="regression",
    return_df=True,
    decimal_places=3,
    include_adjusted_r2=True,
)

print("Test Set Performance Metrics:")

regression_metrics

In [ ]:
coef_df = pd.DataFrame(
    {
        "feature": model_linear_reg.get_feature_names(),
        "coef": model_linear_reg.estimator.named_steps["lr"].coef_,
    }
).sort_values("coef", key=abs, ascending=False)

coef_df.head(20)

In [ ]:
from model_tuner.pickleObjects import loadObjects
from core.functions import resolve_mlflow_run_path
from core.constants import exp_artifact_name, preproc_run_name

artifacts_dir = resolve_mlflow_run_path(
    "../mlruns/preprocessing", exp_artifact_name, preproc_run_name
)

X_columns_list = loadObjects(artifacts_dir / "X_columns_list.pkl")

In [ ]:
numerical_cols = [col for col in X_columns_list if col not in categorical_cols]

## Save Out Predictions from Best Model

In [ ]:
y_test_out = y_test.copy()

In [ ]:
y_test_out.rename(columns={"log_fatalities": "actual_log_fatalities"}, inplace=True)

In [ ]:
X_test.head()

In [ ]:
predictions_x_test = pd.Series(
    model_xgb.predict(X_test), index=X_test.index, name="prediction"
)

In [ ]:
r2_score(
    np.expm1(y_test_out["actual_log_fatalities"]).round(0).astype(int),
    np.expm1(predictions_x_test).round(0).astype(int),
)

In [ ]:
r2_score(y_test_out["actual_log_fatalities"].to_numpy(), predictions_x_test.to_numpy())

In [ ]:
log_preds = predictions_x_test.to_frame(name="log_pred_fatalities")
log_preds["pred_fatalities"] = (
    np.expm1(log_preds["log_pred_fatalities"]).round(0).astype(int)
)

# Use X_test_raw for metadata join
log_preds_merge = log_preds.join(X_test_raw, how="inner")

# Back-transform y_test
log_preds_merge["actual_fatalities"] = (
    np.expm1(y_test_out["actual_log_fatalities"]).round(0).astype(int)
)

log_preds_merge[["actual_fatalities", "pred_fatalities"]]

In [ ]:
log_preds_merge["log_pred_fatalities"]

In [ ]:
from sklearn.metrics import r2_score
import numpy as np

y_actual = np.expm1(
    y_test_out["actual_log_fatalities"]
)  # or np.exp(y_test) depending on how you log-transformed

for name, y_pred_log in zip(
    [
        "Linear Regressor",
        "Lasso RFE",
        "Elastic Net RFE",
        "XGBoost Regressor",
        "CatBoost Regressor",
    ],
    [
        model_linear_reg.predict(X_test_raw),
        model_lasso_rfe.predict(X_test_raw),
        model_elastic_net_rfe.predict(X_test_raw),
        model_xgb.predict(X_test),
        model_cat.predict(X_test),
    ],
):
    y_pred_actual = np.expm1(y_pred_log)  # match the inverse transform
    print(f"{name}: R² (actual scale) = {r2_score(y_actual, y_pred_actual):.3f}")

In [ ]:
from sklearn.metrics import r2_score

r2_score(log_preds_merge["actual_fatalities"], log_preds_merge["pred_fatalities"])

In [ ]:
print(log_preds_merge["actual_fatalities"].describe())
print(
    f"\nEvents with >10 actual fatalities: {(log_preds_merge['actual_fatalities'] > 10).sum()}"
)
print(
    f"Events with >50 actual fatalities: {(log_preds_merge['actual_fatalities'] > 50).sum()}"
)

## Non-Zero Fatalities

In [ ]:
mask = (log_preds_merge["actual_fatalities"] > 0) | (
    log_preds_merge["pred_fatalities"] > 0
)
nonzero = log_preds_merge[mask]

print(f"Non-zero subset: {mask.sum()} of {len(mask)} events ({mask.mean()*100:.1f}%)")
print(
    f"Non-zero R²: {r2_score(nonzero['actual_fatalities'], nonzero['pred_fatalities']):.4f}"
)

In [ ]:
from eda_toolkit import scatter_fit_plot

scatter_fit_plot(
    df=log_preds_merge,
    x_vars=["actual_fatalities"],
    y_vars=["pred_fatalities"],
    show_legend=True,
    show_plot="subplots",
    subplot_figsize=None,
    label_fontsize=14,
    tick_fontsize=12,
    add_best_fit_line=True,
    scatter_color="#808080",
    show_correlation=True,
)

In [ ]:
log_preds_merge.head()

### Save predictions to path

In [ ]:
log_preds_merge.to_csv(
    os.path.join("..", "data", "processed", "log_preds_merge.csv"), index=False
)

## Failure Mode Analysis

In [ ]:
y_test_ground_truth_actual = np.where(y_test["fatalities"] > 0, 1, 0).ravel()

In [ ]:
len(y_test_ground_truth_actual)

In [ ]:
pred_fatalities = log_preds_merge["pred_fatalities"].to_numpy()
pred_fatalities

In [ ]:
show_confusion_matrix(
    y=y_test_ground_truth_actual,
    y_prob=pred_fatalities,
    model_title="XGBoost",
    cmap="Blues",
    text_wrap=50,
    subplots=False,
    n_rows=1,
    figsize=(6, 6),
)

## SHAP values for XGBoost

In [ ]:
# Define a renaming dictionary for clean, paper-ready labels
feature_rename = {
    "cat__source_scale": "Source Scale",
    "cat__interaction": "Interaction Type",
    "cat__sub_event_type": "Sub-Event Type",
    "cat__geo_precision": "Geo Precision",
    "num__days_since_invasion": "Days Since Invasion",
    "num__latitude": "Latitude",
    "cat__admin1": "Admin Region",
    "num__longitude": "Longitude",
    "num__civilian_targeting": "Civilian Targeting",
    "cat__time_precision": "Time Precision",
    "num__percentage_missing": "Percentage Missing",
}


def clean_feature_name(name):
    """Strip prefixes and apply rename mapping, handling expanded category labels."""
    # Check for expanded format: "cat__feature = value"
    if " = " in name:
        prefix_part, value = name.split(" = ", 1)
        base = feature_rename.get(
            prefix_part,
            prefix_part.replace("cat__", "")
            .replace("num__", "")
            .replace("_", " ")
            .title(),
        )
        return f"{base} = {value}"
    # Standard collapsed feature
    return feature_rename.get(
        name, name.replace("cat__", "").replace("num__", "").replace("_", " ").title()
    )

In [ ]:
from core.functions import create_shap_plots
from pathlib import Path
import matplotlib.pyplot as plt

# Define a renaming dictionary for clean, paper-ready labels
feature_rename = {
    "cat__source_scale": "Source Scale",
    "cat__interaction": "Interaction Type",
    "cat__sub_event_type": "Sub-Event Type",
    "cat__geo_precision": "Geo Precision",
    "num__days_since_invasion": "Days Since Invasion",
    "num__latitude": "Latitude",
    "cat__admin1": "Admin Region",
    "num__longitude": "Longitude",
    "num__civilian_targeting": "Civilian Targeting",
    "cat__time_precision": "Time Precision",
    "num__percentage_missing": "Percentage Missing",
}

shap_importance_expanded, shap_importance, shap_figs = create_shap_plots(
    model=model_xgb,
    X_train=X_train,
    X_test=X_test,
    y_test=y_test,
    output_dir=Path(image_path_svg),
    max_display=20,
    sample_size=len(X_test),
    feature_rename=feature_rename,
    side_by_side=True,
    bar_style="summary",
    random_state=222,
)

# Rename features in the returned DataFrames
shap_importance["feature"] = shap_importance["feature"].apply(clean_feature_name)
shap_importance_expanded["feature"] = shap_importance_expanded["feature"].apply(
    clean_feature_name
)

for name, fig in shap_figs.items():
    fig.show()

plt.show()

## Residual Diagnostics

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import pandas as pd
import numpy as np

cat_cols = X_test.select_dtypes(exclude=[np.number]).columns.tolist()
num_cols = X_test.select_dtypes(include=[np.number]).columns.tolist()

ct = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), cat_cols),
])

ct.fit(pd.concat([X_train, X_valid]))
X_test_ohe = ct.transform(X_test)

In [ ]:
# Check what categories the linear model's OHE actually knows about for sub_event_type
ohe_step = model_linear_reg.estimator.named_steps[
    "preprocess_column_transformer_Preprocessor"
].named_transformers_["cat"].named_steps["encoder"]

print(ohe_step.categories_[
    model_linear_reg.estimator.named_steps[
        "preprocess_column_transformer_Preprocessor"
    ].transformers_[1][2].index("sub_event_type")
])

In [ ]:
for name, model in [
    ("Linear Regressor", model_linear_reg),
    ("Lasso RFE", model_lasso_rfe),
    ("Ridge RFE", model_ridge_rfe),
    ("ElasticNet RFE", model_elastic_net_rfe),
    ("XGBoost", model_xgb),
    ("CatBoost", model_cat),
]:
    try:
        n = len(model.get_feature_names())
        print(f"{name}: {n}")
    except Exception as e:
        print(f"{name}: ERROR — {e}")

In [ ]:
linear_features = set(model_linear_reg.get_feature_names())
ohe_features = set(ct.get_feature_names_out())

print("In linear but not OHE:", linear_features - ohe_features)
print("In OHE but not linear:", ohe_features - linear_features)
print(f"Linear: {len(linear_features)}, OHE: {len(ohe_features)}")

In [ ]:
X_train_raw_ohe = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/processed/X_train.parquet"
)
X_valid_raw_ohe = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/processed/X_valid.parquet"
)
X_test_raw_ohe = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/processed/X_test.parquet"
)

ct = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), cat_cols),
])

ct.fit(pd.concat([X_train_raw_ohe, X_valid_raw_ohe]))
X_test_ohe = ct.transform(X_test_raw_ohe)

print(X_test_ohe.shape)

In [ ]:
from model_metrics import show_residual_diagnostics

show_residual_diagnostics(
    y_pred=model_xgb.predict(X_test),
    model_title=["XGBoost Regressor"],
    X=X_test_ohe,
    y=y_test["log_fatalities"],
    n_cols=2,
    plot_type=['fitted', 'qq', 'scale_location', 'histogram'],
    heteroskedasticity_test="breusch_pagan",
    decimal_places=3,
    histogram_type="density",
    kmeans_rstate=222,
    show_diagnostics_table=True,
    image_filename="residual_diagnostics_xgb",
    suptitle="",
    image_path_png=image_path_png,
    image_path_svg=image_path_svg,
)